# C02–M1–L1 — Summarize Private Documents Using RAG, LangChain, and LLMs (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original notebook/files](https://github.com/MohammadWasiq0786/IBM-RAG-and-Agentic-AI/blob/main/02%20Build%20RAG%20Applications/Summarize%20private%20documents%20using%20RAG%20LangChain%20and%20LLMs.ipynb)

**Before you run:** add your free `GOOGLE_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey)) to Colab 🔑 Secrets and enable notebook access. No GPU needed — a CPU runtime is enough.

**What changed vs. the IBM version**
- IBM install + kernel-restart cells, the old Groq/HF `get_llm()` setup and the ~50-line watsonx shim removed → Cell 1 = minimal Gemini install, Cell 2 = universal Gemini init (`llm`, `embeddings`, `llm_model`).
- `Model` + `WatsonxLLM` → `ChatGoogleGenerativeAI`: **flan-ul2 → `gemini-2.5-flash-lite`** (weak-model contrast), **Llama 3.3 70B → `gemini-3.8-flash`**, **Mixtral → `gemini-3.5-flash-lite`**; `GenParams` → `temperature` / `max_output_tokens`.
- `HuggingFaceEmbeddings()` → `GoogleGenerativeAIEmbeddings("models/gemini-embedding-2")`; `Chroma` from `langchain_chroma`.
- `RetrievalQA`, `ConversationalRetrievalChain`, `ConversationBufferMemory` kept (the lab teaches them) via `langchain-classic`; deprecated `qa({...})` calls → `qa.invoke(...)`; `return_message` typo → `return_messages`.
- Colab exercises at the end rewritten for Gemini (embedding-2 vs embedding-001, map_reduce/refine, native LCEL RAG chain, chunk-size sweep, Gemini model comparison).

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai langchain-classic langchain-community langchain-text-splitters wget

### 🔑 Gemini initialization
Run once. Creates `llm` (`gemini-3.8-flash`), `embeddings` (`models/gemini-embedding-2`) and the 1-line `llm_model()` alias used by older lab cells. Needs the `GOOGLE_API_KEY` Colab secret.

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")   # Gemini 3: keep the default temperature (1.0); lower values can cause looping
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
# (.text, not .content: Gemini 3.x can return content as a list of blocks; .text is always a str)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

# **Summarize Private Documents Using RAG, LangChain, and LLMs**


##### Estimated time needed: **45** minutes


Imagine it's your first day at an exciting new job at a fast-growing tech company, Innovatech. You're filled with a mix of anticipation and nerves, eager to make a great first impression and contribute to your team. As you find your way to your desk, decorated with a welcoming note and some company swag, you can't help but feel a surge of pride. This is the moment you've been working towards, and it's finally here.

Your manager, Alex, greets you with a warm smile. "Welcome aboard! We're thrilled to have you with us. I have sent you a folder. Inside this folder, you'll find everything you need to get up to speed on our company policies, culture, and the projects your team is working on. Please keep them private."

You thank Alex and open the folder, only to be greeted by a mountain of documents - manuals, guidelines, technical documents, project summaries, and more. It's overwhelming. You think to yourself, "How am I supposed to absorb all of this information in a short time? And they are private and I cannot just upload it to GPT to summarize them." "Why not create an agent to read and summarize them for you, and then you can just ask it?" your colleague, Jordan, suggests with an encouraging grin. You're intrigued, but uncertain; the world of large language models (LLMs) is one that you've only scratched the surface of. Sensing your hesitation, Jordan elaborates, "Imagine having a personal assistant who's not only exceptionally fast at reading but can also understand and condense the information into easy-to-digest summaries. That's what an LLM can do for you, especially when enhanced with LangChain and Retrieval-Augmented Generation (RAG) technology."  

"But how do I get started? And how long will it take to set up something like that?" you ask. Jordan says, "Let's dive into a project that will not only help you tackle this immediate challenge but also equip you with a skill set that's becoming indispensable in this field."

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/C-rBNv5ZbCn1Qe9a-c_RwQ.png" style="width:50%;margin:auto;display:flex" alt="indexing"/>

---------

So, this project steps you through the fascinating world of LLMs and RAG, starting from the basics of what these technologies are, to building a practical application that can read and summarize documents for you. By the end of this tutorial, you have a working tool capable of processing the pile of documents on your desk, allowing you to focus on making meaningful contributions to your projects sooner.


## __Table of Contents__

<ol>
    <li><a href="#Background">Background</a>
        <ol>
            <li><a href="#What-is-RAG?">What is RAG?</a></li>
            <li><a href="#RAG-architecture">RAG architecture</a></li>
        </ol>
    </li>
    <li>
        <a href="#Objectives">Objectives</a>
    </li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Importing-required-libraries">Importing required libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#Preprocessing">Preprocessing</a>
        <ol>
            <li><a href="#Load-the-document">Load the document</a></li>
            <li><a href="#Splitting-the-document-into-chunks">Splitting the document into chunks</a></li>
            <li><a href="#Embedding-and-storing">Embedding and storing</a></li>
        </ol>
    </li>
    <li>
        <a href="#LLM-model-construction">LLM model construction</a>
    </li>
    <li>
        <a href="#Integrating-LangChain">Integrating LangChain</a>
    </li>
    <li>
        <a href="#Dive-deeper">Dive deeper</a>
        <ol>
            <li><a href="#Using-prompt-template">Using prompt template</a></li>
            <li><a href="#Make-the-conversation-have-memory">Make the conversation have memory</a></li>
            <li><a href="#Wrap-up-and-make-it-an-agent">Wrap up and make it an agent</a></li>
        </ol>
    </li>
</ol>

<a href="#Exercises">Exercises</a>
<ol>
    <li><a href="#Exercise-1:-Work-on-your-own-document">Exercise 1: Work on your own document</a></li>
    <li><a href="#Exercise-2:-Return-the-source-from-the-document">Exercise 2: Return the source from the document</a></li>
    <li><a href="#Exercise-3:-Use-another-LLM-model">Exercise 3: Use another LLM model</a></li>
</ol>


## Background

### What is RAG?
One of the most powerful applications enabled by LLMs is sophisticated question-answering (Q&A) chatbots. These are applications that can answer questions about specific source information. These applications use a technique known as retrieval-augmented generation (RAG). RAG is a technique for augmenting LLM knowledge with additional data, which can be your own data.

LLMs can reason about wide-ranging topics, but their knowledge is limited to public data up to the specific point in time that they were trained. If you want to build AI applications that can reason about private data or data introduced after a model’s cut-off date, you must augment the knowledge of the model with the specific information that it needs. The process of bringing and inserting the appropriate information into the model prompt is known as RAG.

LangChain has several components that are designed to help build Q&A applications and RAG applications, more generally.

### RAG architecture
A typical RAG application has two main components:

* **Indexing**: A pipeline for ingesting and indexing data from a source. This usually happens offline.

* **Retrieval and generation**: The actual RAG chain takes the user query at run time and retrieves the relevant data from the index, then passes that to the model.

The most common full sequence from raw data to answer looks like the following examples.


- **Indexing**
1. Load: First, you must load your data. This is done with [DocumentLoaders](https://python.langchain.com/docs/how_to/#document-loaders).

2. Split: [Text splitters](https://python.langchain.com/docs/how_to/#text-splitters) break large `Documents` into smaller chunks. This is useful both for indexing data and for passing it into a model because large chunks are harder to search and won’t fit in a model’s finite context window.

3. Store: You need somewhere to store and index your splits so that they can later be searched. This is often done using a [VectorStore](https://python.langchain.com/docs/how_to/#vector-stores) and [Embeddings](https://python.langchain.com/docs/how_to/embed_text/) model.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/WEE3pjeJvSZP0R7UL7CYTA.png" width="50%" alt="indexing"/> <br>
<span style="font-size: 10px;">[source](https://python.langchain.com/docs/tutorials/rag/)</span>


- **Retrieval and generation**
1. Retrieve: Given a user input, relevant splits are retrieved from storage using a retriever.
2. Generate: A ChatModel / LLM produces an answer using a prompt that includes the question and the retrieved data.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/SwPO26VeaC8VTZwtmWh5TQ.png" width="50%" alt="retrieval"/> <br>
<span style="font-size: 10px;">[source](https://python.langchain.com/docs/use_cases/question_answering/)</span>


## Objectives

After completing this lab, you will be able to:

 - Master the technique of splitting and embedding documents into formats that LLMs can efficiently read and interpret.
 - Know how to access and utilize different LLMs from IBM watsonx.ai, selecting the optimal model for their specific document processing needs.
 - Implement different retrieval chains from LangChain, tailoring the document retrieval process to support various purposes.
 - Develop an agent that uses integrated LLM, LangChain, and RAG technologies for interactive and efficient document retrieval and summarization, making conversations have memory.


----


## Setup


For this lab, you are going to use the following libraries:

*   [`ibm-watsonx-ai`](https://ibm.github.io/watson-machine-learning-sdk/index.html) for using LLMs from IBM's watsonx.ai
*   [`LangChain`](https://www.langchain.com/) for using its different chain and prompt functions
*   [`Hugging Face`](https://huggingface.co/models?other=embeddings) and [`Hugging Face Hub`](https://huggingface.co/models?other=embeddings) for their embedding methods for processing text data
*   [`SentenceTransformers`](https://www.sbert.net/) for transforming sentences into high-dimensional vectors
*   [`Chroma DB`](https://www.trychroma.com/) for efficient storage and retrieval of high-dimensional text vector data
*   [`wget`](https://pypi.org/project/wget/) for downloading files from remote systems


> **Colab:** `ibm-watsonx-ai` is not installed — the shim above provides `Model`, `WatsonxLLM`, `GenParams` and `DecodingMethods`.

### Importing required libraries
_It is recommended that you import all required libraries in one place (here):_


In [ ]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma                       # was langchain.vectorstores.Chroma
from langchain_core.prompts import PromptTemplate
# Legacy chains/memory taught in this lab live in langchain-classic since LangChain 1.x
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

# Colab (Gemini): ibm_watsonx_ai Model / GenParams / ModelTypes / DecodingMethods and
# WatsonxLLM → ChatGoogleGenerativeAI (imported in Cell 2); HuggingFaceEmbeddings → `embeddings` (Cell 2)
import wget

## Preprocessing
### Load the document

The document, which is provided in a TXT format, outlines some company policies and serves as an example data set for the project.

This is the `load` step in `Indexing`.<br>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MPdUH7bXpHR5muZztZfOQg.png" width="50%" alt="split"/>


In [ ]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Use wget to download the file
wget.download(url, out=filename)
print('file downloaded')

After the file is downloaded and imported into this lab environment, you can use the following code to look at the document.


In [ ]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

From the content, you see that the document discusses nine fundamental policies within a company.


### Splitting the document into chunks


In this step, you are splitting the document into chunks, which is basically the `split` process in `Indexing`.
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/0JFmAV5e_mejAXvCilgHWg.png" width="50%" alt="split"/>


`LangChain` is used to split the document and create chunks. It helps you divide a long story (document) into smaller parts, which are called `chunks`, so that it's easier to handle. 

For the splitting process, the goal is to ensure that each segment is as extensive as if you were to count to a certain number of characters and meet the split separator. This certain number is called `chunk size`. Let's set 1000 as the chunk size in this project. Though the chunk size is 1000, the splitting is happening randomly. This is an issue with LangChain. `CharacterTextSplitter` uses `\n\n` as the default split separator. You can change it by adding the `separator` parameter in the `CharacterTextSplitter` function; for example, `separator="\n"`.


In [ ]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts))

From the ouput of print, you see that the document has been split into 16 chunks


### Embedding and storing
This step is the `embed` and `store` processes in `Indexing`. <br>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/u_oJz3v2cSR_lr0YvU6PaA.png" width="50%" alt="split"/>


In this step, you're taking the pieces of the story, your "chunks," converting the text into numbers, and making them easier for your computer to understand and remember by using a process called "embedding." Think of embedding like giving each chunk its own special code. This code helps the computer quickly find and recognize each chunk later on. 

You do this embedding process during a phase called "Indexing." The reason why is to make sure that when you need to find specific information or details within your larger document, the computer can do so swiftly and accurately.


The following code creates a default embedding model from Hugging Face (**Colab: `models/gemini-embedding-2` from Cell 2**) and ingests them to Chromadb.

When it's completed, print "document ingested".


In [ ]:
# Colab (Gemini): HuggingFaceEmbeddings() → GoogleGenerativeAIEmbeddings("models/gemini-embedding-2") from Cell 2
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb
print('document ingested')

Up to this point, you've been performing the `Indexing` task. The next step is the `Retrieval` task.


## LLM model construction


In this section, you'll build an LLM model from IBM watsonx.ai. 


First, define a model ID and choose which model you want to use. There are many other model options. Refer to [Foundation Models](https://ibm.github.io/watsonx-ai-python-sdk/foundation_models.html) for other model options. This tutorial uses the `FLAN_UL2` model as an example.


In [ ]:
# Colab (Gemini): google/flan-ul2 → the smallest free Gemini model (the "weak model" of the lab)
model_id = 'gemini-2.5-flash-lite'

Define parameters for the model.

The decoding method is set to `greedy` to get a deterministic output.

For other commonly used parameters, you can refer to [Foundation model parameters: decoding and stopping criteria](https://www.ibm.com/docs/en/watsonx-as-a-service?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-RAG_v1_1711546843&topic=lab-model-parameters-prompting).


In [ ]:
# Colab (Gemini): GenParams → ChatGoogleGenerativeAI settings
# (DECODING_METHOD greedy ≈ temperature 0; MIN_NEW_TOKENS has no Gemini equivalent)
parameters = {
    "max_output_tokens": 256,  # this controls the maximum number of tokens in the generated output
    "temperature": 0.5 # this randomness or creativity of the model's responses
}

Define `credentials` and `project_id`,  which are necessary parameters to successfully run LLMs from watsonx.ai.

(Keep `credentials` and `project_id` as they are now so that you do not need to create your own keys to run models. This supports you in running the model inside this lab environment. However, if you want to run the model locally, refer to this [tutorial](https://medium.com/the-power-of-ai/ibm-watsonx-ai-the-interface-and-api-e8e1c7227358) for creating your own keys.


> **Colab (Gemini):** no watsonx account, `credentials` or `project_id` needed — the Gemini key comes from Colab Secrets (Cell 2). The original "API Disclaimer / Running Locally" text is not relevant here.

In [ ]:
# Colab (Gemini): watsonx credentials/project_id are not needed — GOOGLE_API_KEY was set in Cell 2
credentials = None
project_id = None

Wrap the parameters to the model.


In [ ]:
# Colab (Gemini): Model(model_id, params, credentials, project_id) → ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model=model_id, **parameters)

Build a model called `flan_ul2_llm` from watsonx.ai.


In [ ]:
flan_ul2_llm = model   # was WatsonxLLM(model=model); the name is kept so later cells run unchanged

This completes the `LLM` part of the `Retrieval` task. <br>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UZXQ44Tgv4EQ2-mTcu5e-A.png" width="50%" alt="split"/>


## Integrating LangChain


LangChain has a number of components that are designed to help retrieve information from the document and build question-answering applications, which helps you complete the `retrieve` part of the `Retrieval` task. <br>
<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/M4WpkkMMbfK0Wkz0W60Jiw.png" width="50%" alt="split"/>


In the following steps, you create a simple Q&A application over the document source using LangChain's `RetrievalQA`.

Then, you ask the query "what is mobile policy?"


In [ ]:
qa = RetrievalQA.from_chain_type(llm=flan_ul2_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "what is mobile policy?"
qa.invoke(query)

From the response, it seems fine. The model's response is the relevant information about the mobile policy from the document.


Now, try to ask a more high-level question.


In [ ]:
qa = RetrievalQA.from_chain_type(llm=flan_ul2_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

At this time, the model seems to not have the ability to summarize the document. This is because of the limitation of the `FLAN_UL2` model.


> **Colab (Gemini):** flan-ul2 is mapped to `gemini-2.5-flash-lite` with only 256 output tokens. It *can* summarize, so the contrast is weaker than in the original — compare its answer length and coverage with the bigger model below.

So, try to use another model, `LLAMA_3_70B_INSTRUCT` (**Colab: `gemini-3.8-flash`**). You should do the model construction again.

In [ ]:
# Colab (Gemini): meta-llama/llama-3-3-70b-instruct → gemini-3.8-flash
model_id = 'gemini-3.8-flash'

parameters = {
    "max_output_tokens": 1024,  # headroom for Gemini thinking tokens
    "temperature": 0.5 # this randomness or creativity of the model's responses
}

model = ChatGoogleGenerativeAI(model=model_id, **parameters)

llama_3_llm = model   # was WatsonxLLM(model=model)

Try the same query again on this model.


In [ ]:
qa = RetrievalQA.from_chain_type(llm=llama_3_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can you summarize the document for me?"
qa.invoke(query)

Now, you've created a simple Q&A application for your own document. Congratulations!


In [ ]:
ans= qa.invoke(query)

print(ans['query'])
print("-"*50)
print(ans['result'])

## Dive deeper


This section dives deeper into how you can improve this application. You might want to ask "How to add the prompt in retrieval using LangChain?" <br>

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/bvw3pPRCYRUsv-Z2m33hmQ.png" width="50%" alt="split"/>


You use prompts to guide the responses from an LLM the way you want. For instance, if the LLM is uncertain about an answer, you instruct it to simply state, "I do not know," instead of attempting to generate a speculative response.

Let's see an example.


In [ ]:
qa = RetrievalQA.from_chain_type(llm=flan_ul2_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "Can I eat in company vehicles?"
qa.invoke(query)

As you can see, the query is asking something that does not exist in the document. The LLM responds with information that actually is not true. You don't want this to happen, so you must add a prompt to the LLM.


### Using prompt template


In the following code, you create a prompt template using `PromptTemplate`.

`context` and `question` are keywords in the RetrievalQA, so LangChain can automatically recognize them as document content and query.


In [ ]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

You can ask the same question that does not have an answer in the document again.


In [ ]:
qa = RetrievalQA.from_chain_type(llm=llama_3_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

From the answer, you can see that the model responds with "don't know".


### Make the conversation have memory


Do you want your conversations with an LLM to be more like a dialogue with a friend who remembers what you talked about last time? An LLM that retains the memory of your previous exchanges builds a more coherent and contextually rich conversation.


Take a look at a situation in which an LLM does not have memory.

You start a new query, "What I cannot do in it?". You do not specify what "it" is. In this case, "it" means "company vehicles" if you refer to the last query.


In [ ]:
query = "What I cannot do in it?"
qa.invoke(query)

From the response, you see that the model does not have the memory because it does not provide the correct answer, which is something related to "smoking is not permitted in company vehicles."


To make the LLM have memory, you introduce the `ConversationBufferMemory` function from LangChain.


In [ ]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_messages = True)

Create a `ConversationalRetrievalChain` to retrieve information and talk with the LLM.


In [ ]:
qa = ConversationalRetrievalChain.from_llm(llm=llama_3_llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory, 
                                           get_chat_history=lambda h : h, 
                                           return_source_documents=False)

Create a `history` list to store the chat history.


In [ ]:
history = []

In [ ]:
query = "What is mobile policy?"
result = qa.invoke({"question": query})   # memory supplies chat_history
print(result["answer"])

Append the previous query and answer to the history.


In [ ]:
history.append((query, result["answer"]))

In [ ]:
query = "List points in it?"
result = qa.invoke({"question": query})   # memory supplies chat_history
print(result["answer"])

Append the previous query and answer to the chat history again.


In [ ]:
history.append((query, result["answer"]))

In [ ]:
query = "What is the aim of it?"
result = qa.invoke({"question": query})   # memory supplies chat_history
print(result["answer"])

### Wrap up and make it an agent


The following code defines a function to make an agent, which can retrieve information from the document and has the conversation memory.


In [ ]:
def qa():
    memory = ConversationBufferMemory(memory_key = "chat_history", return_messages = True)
    qa = ConversationalRetrievalChain.from_llm(llm=llama_3_llm, 
                                               chain_type="stuff", 
                                               retriever=docsearch.as_retriever(), 
                                               memory = memory, 
                                               get_chat_history=lambda h : h, 
                                               return_source_documents=False)
    history = []
    while True:
        query = input("Question: ")
        
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break
            
        result = qa.invoke({"question": query})
        
        history.append((query, result["answer"]))
        
        print("Answer: ", result["answer"])

Run the function.

Feel free to answer questions for your chatbot. For example: 

_What is the smoking policy? Can you list all points of it? Can you summarize it?_

To **stop** the agent, you can type in 'quit', 'exit', 'bye'. Otherwise you cannot run other cells. 


In [ ]:
qa()

Congratulations! You have finished the project. Following are three exercises to help you to extend your knowledge.


# Exercises


> 💡 **How to use the ✅ Answer cells:** each exercise is followed by a hidden **✅ Answer** cell (form view, code collapsed). Try the exercise first. To check, tick `run_answer` in the form (or double-click to show the code and set `run_answer = True`) and run the cell. With `run_answer` unticked, the cell stops with *Answer not run* — so **Runtime → Run all** stops at the first answer cell; run the cells below it one by one.

### Exercise 1: Work on your own document


You are welcome to use your own document to practice. Another document has also been prepared that you can use for practice. Can you load this document and make the LLM read it for you? <br>
Here is the URL to the document: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt


In [ ]:
# Add your code here

filename = 'stateOfUnion.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt'

wget.download(url, out=filename)
print('file downloaded')

<details>
    <summary>Click here for solution</summary>
<br>
    
```python
filename = 'stateOfUnion.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt'

wget.download(url, out=filename)
print('file downloaded')
```

</details>


In [ ]:
# @title ✅ Answer — Exercise 1: Work on your own document
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

filename = 'stateOfUnion.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt'
wget.download(url, out=filename)
print('file downloaded')

# Same pipeline as the lab: load → split → embed/store → RetrievalQA
sotu_docs = TextLoader(filename).load()
sotu_texts = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_documents(sotu_docs)
print(len(sotu_texts), "chunks")
# Own collection: without collection_name it would share Chroma's default "langchain" collection
sotu_search = Chroma.from_documents(sotu_texts, embeddings, collection_name="state_of_union",
                                    ids=[f"sotu-{i}" for i in range(len(sotu_texts))])
qa_sotu = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type="stuff",
                                      retriever=sotu_search.as_retriever(),
                                      chain_type_kwargs=chain_type_kwargs,   # the "say you don't know" prompt
                                      return_source_documents=False)
for q in ["Can you summarize the document for me?",
          "What does the speech say about Russia and Ukraine?",
          "What is the company's mobile phone policy?"]:      # not in this document → "don't know"
    print("\nQ:", q, "\nA:", qa_sotu.invoke(q)["result"])

# Expected / what you should see: `file downloaded`, `42 chunks`, a summary of the 2022 State of the Union (infrastructure, inflation, Ukraine…), a Russia/Ukraine answer, and **"I don't know"** for the mobile-phone question (it is not in this document).


### Exercise 2: Return the source from the document


Sometimes, you not only want the LLM to summarize for you, but you also want the model to return the exact content source from the document to you for reference. Can you adjust the code to make it happen?


In [ ]:
# Add your code 
qa = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type="stuff", retriever=docsearch.as_retriever(), return_source_documents=True)
query = "Can I smoke in company vehicles?"
results = qa.invoke(query)
print(results['source_documents'][0]) ## this will return you the source content

<details>
    <summary>Click here for a hint</summary>
All you must do is change the return_source_documents to True when you create the chain. And when you print, print the ['source_documents'][0] 
<br><br>

    
```python
qa = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type="stuff", retriever=docsearch.as_retriever(), return_source_documents=True)
query = "Can I smoke in company vehicles?"
results = qa.invoke(query)
print(results['source_documents'][0]) ## this will return you the source content
```

</details>


In [ ]:
# @title ✅ Answer — Exercise 2: Return the source from the document
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

qa = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type="stuff",
                                 retriever=docsearch.as_retriever(),
                                 return_source_documents=True)      # ← the only change
query = "Can I smoke in company vehicles?"
results = qa.invoke(query)
print("Answer:", results["result"])
print("-" * 50)
print(results["source_documents"][0])                 # the exact source chunk
for i, d in enumerate(results["source_documents"]):   # all retrieved chunks
    print(f"[{i}] {d.metadata.get('source')}: {d.page_content[:80]!r}")

# Expected / what you should see: `Answer:` saying smoking is not allowed in company vehicles, then the source `Document` (metadata `source='companyPolicies.txt'`) whose text contains *5. Smoking Policy*.


### Exercise 3: Use another LLM model


IBM watsonx.ai also has many other LLM models that you can use; for example, `mistralai/mixtral-8x7b-instruct-v01`, an open-source model from Mistral AI. Can you change the model to see the difference of the response?


> **Colab (Gemini):** Mixtral → `gemini-3.5-flash-lite`. Any model from the free list works, e.g. `gemini-2.5-flash` or `gemini-3.1-flash-lite`.

In [ ]:
# Add your code here
# Colab (Gemini): mistralai/mixtral-8x7b-instruct-v01 → gemini-3.5-flash-lite
model_id = 'gemini-3.5-flash-lite'

parameters = {
    "max_output_tokens": 1024,
    "temperature": 0.5 # this randomness or creativity of the model's responses
}

mistral_llm = ChatGoogleGenerativeAI(model=model_id, **parameters)

# Add your code 
qa = RetrievalQA.from_chain_type(llm=mistral_llm, chain_type="stuff", retriever=docsearch.as_retriever(), return_source_documents=True)
query = "Can I smoke in company vehicles?"
results = qa.invoke(query)
print(results['result'])
print(results['source_documents'][0]) ## this will return you the source content

<details>
    <summary>Click here for a hint</summary>

To use a different LLM, go to the cell where the `model_id` is specified and replace the current `model_id` with the following code. Expect different results and performance when using different LLMs: 

```python
model_id = 'mistralai/mixtral-8x7b-instruct-v01'   # Colab: 'gemini-3.5-flash-lite'
```
</br>

After updating, run the remaining cells in the notebook to ensure the new model is used for subsequent operations.

</details>


In [ ]:
# @title ✅ Answer — Exercise 3: Use another LLM model
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

# mistralai/mixtral-8x7b-instruct-v01 → gemini-3.5-flash-lite (Gemini 3: default temperature, minimal thinking by default)
model_id = 'gemini-3.5-flash-lite'
parameters = {"max_output_tokens": 1024}
mistral_llm = ChatGoogleGenerativeAI(model=model_id, **parameters)

query = "Can I smoke in company vehicles?"
for name, model in [("gemini-3.8-flash (llama_3_llm)", llama_3_llm), ("gemini-3.5-flash-lite (mistral_llm)", mistral_llm)]:
    qa = RetrievalQA.from_chain_type(llm=model, chain_type="stuff",
                                     retriever=docsearch.as_retriever(), return_source_documents=True)
    results = qa.invoke(query)
    print(f"\n== {name}\n{results['result']}")
    print("source:", results["source_documents"][0].page_content[:100].replace("\n", " "))

# Expected / what you should see: Two answers to the same question. `gemini-3.5-flash-lite` is faster and shorter; `gemini-3.8-flash` gives more detail. Both should say smoking in company vehicles is not allowed.


---
# 🧪 Colab practice exercises
Extra practice on top of the lab's own exercises. Each cell has a starter; complete the `TODO`s.

### C1 · `gemini-embedding-2` vs `gemini-embedding-001`
Rebuild the Chroma store with the older Gemini embedding model and compare the top-3 chunks for the same query.

In [ ]:
emb_001 = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db_001 = Chroma.from_documents(texts, emb_001, collection_name="emb001")
q = "What is the policy on smoking?"
for name, db in [("gemini-embedding-2 (default)", docsearch), ("gemini-embedding-001", db_001)]:
    print(f"\n== {name}")
    for d in db.similarity_search(q, k=3):
        print("-", d.page_content[:120].replace("\n", " "))
# TODO: try 3 more queries — which model ranks the right policy first more often?

In [ ]:
# @title ✅ Answer — C1 · gemini-embedding-2 vs gemini-embedding-001
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

# For each query: which policy title does each store rank first? (expected = the policy in the right column)
tests = [("What is the policy on smoking?", "Smoking"),
         ("Can I use my phone at work?", "Mobile Phone"),
         ("How are new employees hired?", "Recruitment"),
         ("What happens if I break the rules?", "Discipline"),
         ("Is harassment tolerated?", "Anti-discrimination")]
score = {"gemini-embedding-2": 0, "gemini-embedding-001": 0}
for q, expected in tests:
    row = []
    for name, db in [("gemini-embedding-2", docsearch), ("gemini-embedding-001", db_001)]:
        top = db.similarity_search(q, k=1)[0].page_content
        hit = expected.lower() in top.lower()
        score[name] += hit
        row.append(f"{name}: {'✅' if hit else '❌'} {top[:50]!r}")
    print(f"\n{q}\n  " + "\n  ".join(row))
print("\nTop-1 hits:", score)

# Expected / what you should see: A ✅/❌ per query for each embedding model. Both usually rank Smoking / Mobile Phone first; the 4th (vague) query is where they differ.


### C2 · Summarize with `map_reduce` and `refine`
`stuff` puts every retrieved chunk in one prompt. Try the other chain types on the summary question and compare length/quality.

In [ ]:
for ct in ["stuff", "map_reduce", "refine"]:
    qa_ct = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type=ct,
                                        retriever=docsearch.as_retriever(search_kwargs={"k": 6}))
    print(f"\n== {ct}\n", qa_ct.invoke("Can you summarize the document for me?")["result"][:800])
# TODO: count LLM calls per chain type (hint: langchain_core.callbacks + a counter, or time each run).

In [ ]:
# @title ✅ Answer — C2 · Summarize with map_reduce and refine
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

from langchain_core.callbacks import BaseCallbackHandler

class LLMCallCounter(BaseCallbackHandler):
    def __init__(self): self.calls = 0
    def on_chat_model_start(self, serialized, messages, **kwargs): self.calls += 1
    def on_llm_start(self, serialized, prompts, **kwargs): self.calls += 1

import time
for ct in ["stuff", "map_reduce", "refine"]:
    counter = LLMCallCounter()
    qa_ct = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type=ct,
                                        retriever=docsearch.as_retriever(search_kwargs={"k": 6}))
    t = time.time()
    out = qa_ct.invoke("Can you summarize the document for me?", config={"callbacks": [counter]})["result"]
    print(f"== {ct}: {counter.calls} LLM calls · {time.time()-t:.1f}s · {len(out)} chars")
    time.sleep(1)  # be gentle with the free-tier rate limit

# Expected / what you should see: `stuff` = 1 LLM call; `map_reduce` ≈ k+1 calls (7 with k=6); `refine` = k calls (6). `stuff` is fastest; `refine` is usually the longest summary.


### C3 · Rewrite the QA chain with LCEL
`RetrievalQA` is legacy. Build the same thing as a native LCEL chain: `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = docsearch.as_retriever(search_kwargs={"k": 4})
rag_prompt = ChatPromptTemplate.from_template("""Answer only from the context. If the answer is not there, say you don't know.

{context}

Question: {question}""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | rag_prompt | llm | StrOutputParser())
print(rag_chain.invoke("Can I eat in company vehicles?"))
# TODO: add chat history — rewrite the follow-up "What I cannot do in it?" into a standalone
# question with a small `condense_prompt | llm | StrOutputParser()` chain before retrieval.

In [ ]:
# @title ✅ Answer — C3 · Rewrite the QA chain with LCEL
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

condense_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's last question as a standalone question, using the chat history. "
               "Return only the question."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])
condense_chain = condense_prompt | llm | StrOutputParser()

def ask(question, chat_history):
    standalone = condense_chain.invoke({"chat_history": chat_history, "question": question}) if chat_history else question
    answer = rag_chain.invoke(standalone)          # rag_chain from the starter cell
    chat_history += [HumanMessage(question), AIMessage(answer)]
    return standalone, answer

chat_history = []
for q in ["Can I smoke in company vehicles?", "What I cannot do in it?"]:
    standalone, answer = ask(q, chat_history)
    print(f"Q: {q}\n  standalone: {standalone}\n  A: {answer}\n")

# Expected / what you should see: The LCEL chain says it doesn't know about eating in vehicles. With history, the follow-up is rewritten to a standalone question (e.g. *What can't I do in company vehicles?*) and answered from the Smoking policy.


### C4 · Chunk-size experiment
Split with `RecursiveCharacterTextSplitter` at different sizes and see how retrieval changes.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
for size in [200, 500, 1000]:
    chunks = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10).split_documents(documents)
    db = Chroma.from_documents(chunks, embeddings, collection_name=f"cs{size}")
    top = db.similarity_search("What is the mobile phone policy?", k=1)[0].page_content
    print(f"chunk_size={size}: {len(chunks)} chunks · top hit: {top[:100]!r}")
# TODO: which size gives the best answer from qa.invoke? Why?

In [ ]:
# @title ✅ Answer — C4 · Chunk-size experiment
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

from langchain_text_splitters import RecursiveCharacterTextSplitter
question = "What is the mobile phone policy?"
for size in [200, 500, 1000]:
    chunks = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10).split_documents(documents)
    db = Chroma.from_documents(chunks, embeddings, collection_name=f"cs{size}",
                               ids=[f"cs{size}-{i}" for i in range(len(chunks))])
    qa_cs = RetrievalQA.from_chain_type(llm=llama_3_llm, chain_type="stuff",
                                        retriever=db.as_retriever(search_kwargs={"k": 4}))
    print(f"\n== chunk_size={size}: {len(chunks)} chunks\n", qa_cs.invoke(question)["result"][:500])

# Expected / what you should see: 200 → ~122 chunks, 500 → ~50, 1000 → ~26. Small chunks retrieve only fragments; 500–1000 gives the most complete answer for the mobile-phone policy.


### C5 · Compare free Gemini models
Run the same RAG question with `gemini-3.8-flash`, `gemini-3.5-flash-lite` and `gemini-2.5-flash` (the swap-rule model comparison).

In [ ]:
import time
for name in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-2.5-flash"]:
    chat = ChatGoogleGenerativeAI(model=name, temperature=0)
    chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | rag_prompt | chat | StrOutputParser())
    t = time.time(); ans = chain.invoke("Can you summarize the document for me?")
    print(f"\n== {name} ({time.time()-t:.1f}s)\n", ans[:600])
# TODO: note answer quality and speed for each model in a small table.

In [ ]:
# @title ✅ Answer — C5 · Compare free Gemini models
run_answer = False  # @param {type:"boolean"}
if not run_answer:
    class AnswerNotRun(Exception):
        def _render_traceback_(self):
            return ["⏸ Answer not run — tick run_answer (or set it to True) and run this cell again."]
    raise AnswerNotRun("Answer not run")

import time
rows = []
for name in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash"]:
    chat = ChatGoogleGenerativeAI(model=name)   # default temperature (Gemini 3 recommendation)
    chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | rag_prompt | chat | StrOutputParser())
    t = time.time()
    try:
        ans = chain.invoke("Can you summarize the document for me?")
        rows.append((name, f"{time.time()-t:.1f}s", len(ans), ans[:80].replace("\n", " ")))
    except Exception as e:      # 404 / access refused (2.5 models on new keys) / 429 free-tier limit
        rows.append((name, "-", 0, f"⚠️ {type(e).__name__}"))
    time.sleep(2)

print(f"{'model':24} {'time':>6} {'chars':>6}  first words")
for r in rows:
    print(f"{r[0]:24} {r[1]:>6} {r[2]:>6}  {r[3]}")

# Expected / what you should see: A small table (model, time, chars). 3.x models answer; `gemini-2.5-flash` may show `⚠️ … 404/403` for new API keys or `429` on the free tier — that's expected.


## Authors


[Kang Wang](https://author.skills.network/instructors/kang_wang) <br>
Kang Wang is a Data Scientist Intern in IBM. He is also a PhD Candidate in the University of Waterloo.

[Faranak Heidari](https://www.linkedin.com/in/faranakhdr/) <br>
Faranak Heidari is a Data Scientist Intern in IBM with a strong background in applied machine learning. Experienced in managing complex data to establish business insights and foster data-driven decision-making in complex settings such as healthcare. She is also a PhD candidate at the University of Toronto.


### Other Contributors


[Sina Nazeri](https://author.skills.network/instructors/sina_nazeri) <br>
I am grateful to have had the opportunity to work as a Research Associate, Ph.D., and IBM Data Scientist. Through my work, I have gained experience in unraveling complex data structures to extract insights and provide valuable guidance.

[Wojciech "Victor" Fulmyk](https://author.skills.network/instructors/wojciech_fulmyk) <br>
Wojciech "Victor" Fulmyk is a Data Scientist at IBM and a Ph.D. candidate in Economics at the University of Calgary.


```{## Change Log}
```


```{|Date (YYYY-MM-DD)|Version|Changed By|Change Description||-|-|-|-||2024-03-22|0.1|Kang Wang|Create the Project|}
```


© Copyright IBM Corporation. All rights reserved.
